<a href="https://colab.research.google.com/github/VictorMelendez4/Proyecto_MLB_DataScience/blob/main/Proyecto_MLB_DataScience.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Instalando PySpark...")
!pip install -q pyspark

print("Encendiendo el motor...")
from pyspark.sql import SparkSession

try:
    # Usamos la configuración nativa de Colab sin alterar variables de entorno
    spark = SparkSession.builder.master("local[*]").appName("ProyectoMLB_Final").getOrCreate()
    print("MOTOR ENCENDIDO Listo para la Fase 1.")
except Exception as e:
    print(f" Error: {e}")

Instalando PySpark...
Encendiendo el motor...
MOTOR ENCENDIDO Listo para la Fase 1.


In [2]:
# --- FASE 1 ---

!pip install -q findspark
import os
import findspark

print("Configurando el entorno de Spark...")

# Aseguramos que JAVA_HOME y SPARK_HOME estén correctamente definidos
if "SPARK_HOME" in os.environ:
    del os.environ["SPARK_HOME"]
if "JAVA_HOME" in os.environ:
    del os.environ["JAVA_HOME"]

# Establecemos SPARK_HOME a la ruta donde se encuentra Spark en Colab
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"

# Inicializamos findspark para que PySpark pueda encontrar la instalación
findspark.init()

print("Encendiendo el motor con la instalación limpia...")
from pyspark.sql import SparkSession

try:
    spark = SparkSession.builder.master("local[*]").appName("ProyectoMLB_Rapido").getOrCreate()
    print("Spark está listo y operativo.")
except Exception as e:
    print(f"Sigue habiendo un error: {e}")

Configurando el entorno de Spark...
Encendiendo el motor con la instalación limpia...
Spark está listo y operativo.


In [3]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

In [4]:
import findspark
findspark.init()

In [5]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("ProyectoMLB_Colab").getOrCreate()

In [6]:
from google.colab import userdata

try:
    # Intentamos leer el secreto que acabas de crear
    mi_llave = userdata.get('API_BALL_DONT_LIE')
    print("¡Excelente! Colab logró leer tu clave secreta.")
    print("Tu clave empieza con:", mi_llave[:4] + "...") # Solo mostramos los primeros 4 caracteres por seguridad
except userdata.SecretNotFoundError:
    print("Error: No se encontró el secreto. Revisa si le pusiste bien el Nombre.")
except userdata.NotebookAccessError:
    print("Error: El secreto existe, pero olvidaste darle clic al botoncito para que se ponga azul (Acceso desde el notebook).")

¡Excelente! Colab logró leer tu clave secreta.
Tu clave empieza con: 1864...


In [7]:
import requests
import time
import json

from pyspark.sql import SparkSession
from google.colab import userdata

In [9]:
import requests
import json
from google.colab import userdata
from pyspark.sql import functions as F

# Configuración de Acceso
api_key = userdata.get('API_BALL_DONT_LIE')
headers = {"Authorization": api_key}

# Saltamos a la página 15 (aprox 1,500 juegos adentro) para asegurar temporada regular
url = "https://api.balldontlie.io/mlb/v1/games?seasons[]=2026&season_type=regular&per_page=100"
print("Descargando datos desde el proveedor...")
response = requests.get(url, headers=headers)

if response.status_code == 200:
    datos_json = response.json()['data']

    if len(datos_json) > 0:
        # Conversión a DataFrame
        df_raw = spark.read.json(spark.sparkContext.parallelize([json.dumps(d) for d in datos_json]))
        print(f" Ingesta completada: {df_raw.count()} registros cargados.\n")

        # verificamos en que meses estamos ubicados
        print("Rango de fechas de los datos descargados:")
        df_raw.select(
            F.min("date").alias("Juego_Mas_Antiguo"),
            F.max("date").alias("Juego_Mas_Reciente")
        ).show(truncate=False)

    else:
        print(" La API devolvió 0 juegos. Nos pasamos de página (el futuro aún no se juega).")
else:
    print(f" Error en comunicación con API: {response.status_code} - {response.text}")

Descargando datos desde el proveedor...
 Ingesta completada: 100 registros cargados.

Rango de fechas de los datos descargados:
+------------------------+------------------------+
|Juego_Mas_Antiguo       |Juego_Mas_Reciente      |
+------------------------+------------------------+
|2026-03-26T00:05:00.000Z|2026-04-03T18:10:00.000Z|
+------------------------+------------------------+



In [11]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer

In [12]:
# --- FASE 2 ---
df_clean = df_raw

# Corrección de campos y extracción PROFUNDA (Runs, Hits y Errors)
df_clean = df_clean.withColumn("score_home", F.col("home_team_data").getItem("runs").cast("integer")) \
                   .withColumn("score_away", F.col("away_team_data").getItem("runs").cast("integer")) \
                   .withColumn("hits_home", F.col("home_team_data").getItem("hits").cast("integer")) \
                   .withColumn("hits_away", F.col("away_team_data").getItem("hits").cast("integer")) \
                   .withColumn("errores_home", F.col("home_team_data").getItem("errors").cast("integer")) \
                   .withColumn("errores_away", F.col("away_team_data").getItem("errors").cast("integer"))

# Llenamos nulos por si acaso
df_clean = df_clean.fillna(0, subset=["score_home", "score_away", "hits_home", "hits_away", "errores_home", "errores_away"])

# Label y filtro de nulos (lo mantenemos simple)
df_clean = df_clean.withColumn("label", F.when(F.col("score_home") > F.col("score_away"), 1).otherwise(0))

# Indexamos equipos
indexer_home = StringIndexer(inputCol="home_team_name", outputCol="home_team_id", handleInvalid="keep")
indexer_away = StringIndexer(inputCol="away_team_name", outputCol="away_team_id", handleInvalid="keep")

df_ml = indexer_home.fit(df_clean).transform(df_clean)
df_ml = indexer_away.fit(df_ml).transform(df_ml)

print(f" Datos listos: Procesados {df_ml.count()} juegos en total (pasado + futuro).")

 Datos listos: Procesados 100 juegos en total (pasado + futuro).


In [13]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql.functions import rand, when

In [14]:
# FASE 2.5: FEATURE ENGINEERING

from pyspark.sql.window import Window
import pyspark.sql.functions as F

print(" Generando Inteligencia de Negocio...")

# 1. Ventanas de Tiempo (Evitando Data Leakage)
window_home = Window.partitionBy("home_team_id").orderBy("date").rowsBetween(Window.unboundedPreceding, -1)
window_away = Window.partitionBy("away_team_id").orderBy("date").rowsBetween(Window.unboundedPreceding, -1)

# 2. Cálculo de Promedios Históricos (Carreras, Hits y Errores encadenados)
df_features = df_ml.withColumn("Ofensiva_Local", F.avg("score_home").over(window_home)) \
                   .withColumn("Ofensiva_Visitante", F.avg("score_away").over(window_away)) \
                   .withColumn("Defensa_Local", F.avg("score_away").over(window_home)) \
                   .withColumn("Defensa_Visitante", F.avg("score_home").over(window_away)) \
                   .withColumn("Promedio_Hits_Local", F.avg("hits_home").over(window_home)) \
                   .withColumn("Promedio_Hits_Visitante", F.avg("hits_away").over(window_away)) \
                   .withColumn("Promedio_Errores_Local", F.avg("errores_home").over(window_home)) \
                   .withColumn("Promedio_Errores_Visitante", F.avg("errores_away").over(window_away))

# 3. Limpieza de Nulos y Redondeo
metricas = [
    "Ofensiva_Local", "Ofensiva_Visitante", "Defensa_Local", "Defensa_Visitante",
    "Promedio_Hits_Local", "Promedio_Hits_Visitante", "Promedio_Errores_Local", "Promedio_Errores_Visitante"
]

#llenamos nulos en el DataFrame completo
df_features = df_features.fillna(0, subset=metricas)

#redondeamos todo a 2 decimales
for col_name in metricas:
    df_features = df_features.withColumn(col_name, F.round(F.col(col_name), 2))

print("Variables predictivas generadas con éxito.")
df_features.select("date", "home_team_name", "Ofensiva_Local", "Promedio_Hits_Local", "Promedio_Errores_Local").show(5)

 Generando Inteligencia de Negocio...
Variables predictivas generadas con éxito.
+--------------------+--------------+--------------+-------------------+----------------------+
|                date|home_team_name|Ofensiva_Local|Promedio_Hits_Local|Promedio_Errores_Local|
+--------------------+--------------+--------------+-------------------+----------------------+
|2026-03-26T20:10:...|Houston Astros|           0.0|                0.0|                   0.0|
|2026-03-28T00:10:...|Houston Astros|           0.0|                3.0|                   0.0|
|2026-03-28T23:10:...|Houston Astros|           1.0|                5.5|                   0.0|
|2026-03-29T18:10:...|Houston Astros|          4.33|                8.0|                   0.0|
|2026-03-31T00:10:...|Houston Astros|           5.5|               7.75|                   0.0|
+--------------------+--------------+--------------+-------------------+----------------------+
only showing top 5 rows


In [16]:
# FASE 3 Y 4
# se entrena el modelo Random Forest y ejecutar el diagnóstico de riesgos.

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.functions import vector_to_array
import pyspark.sql.functions as F

print("Iniciando Fase 3: Entrenamiento de Inteligencia Artificial (Random Forest)...\n")

# Ensamblaje del Vector de Características
assembler = VectorAssembler(
    inputCols=[
        "home_team_id", "away_team_id",
        "Ofensiva_Local", "Ofensiva_Visitante",
        "Defensa_Local", "Defensa_Visitante",
        "Promedio_Hits_Local", "Promedio_Hits_Visitante", # ¡NUEVO!
        "Promedio_Errores_Local", "Promedio_Errores_Visitante" # ¡NUEVO!
    ],
    outputCol="features"
)
df_model_completo = assembler.transform(df_features)

# Filtrado y División de Datos (Data Splitting)
# Excluimos posibles juegos sin terminar o errores de marcador
df_para_entrenar = df_model_completo.filter((F.col("status") == "STATUS_FINAL") & ((F.col("score_home") + F.col("score_away")) > 0))

# Hold-out validation (80% Entrenamiento, 20% Prueba)
train_data, test_data = df_para_entrenar.randomSplit([0.8, 0.2], seed=42)
print(f"Dataset de Entrenamiento: {train_data.count()} juegos históricos.")
print(f"Dataset de Validación: {test_data.count()} juegos.\n")

#Entrenamiento del Algoritmo
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=30, maxDepth=5, maxBins=40)
print("Ajustando hiperparámetros y entrenando el modelo...")
modelo_entrenado = rf.fit(train_data)

# Inferencia y Análisis de Riesgos
print("Extrayendo probabilidades y calculando niveles de riesgo...\n")
predicciones = modelo_entrenado.transform(test_data)

# Desglose de probabilidades y reglas de negocio (Semáforo de Riesgo y Anomalías)
df_analisis = predicciones.withColumn("prob_array", vector_to_array("probability")) \
    .withColumn("Prob_Visitante_%", F.round(F.col("prob_array")[0] * 100, 2)) \
    .withColumn("Prob_Local_%", F.round(F.col("prob_array")[1] * 100, 2)) \
    .withColumn("Nivel_Riesgo",
        F.when((F.col("Prob_Local_%") >= 40) & (F.col("Prob_Local_%") <= 60), "Alta Volatilidad (Evitar)")
         .otherwise("Favorito Matemático")) \
    .withColumn("Anomalia_Deportiva",
        F.when((F.col("label") != F.col("prediction")) & ((F.col("Prob_Local_%") >= 65) | (F.col("Prob_Visitante_%") >= 65)), "ROMPE-QUINIELAS")
         .otherwise("Comportamiento Normal"))

# Reporte
columnas_vista = [
    "date", "home_team_name", "score_home", "score_away", "away_team_name",
    "Prob_Local_%", "Prob_Visitante_%", "Nivel_Riesgo", "Anomalia_Deportiva"
]

print("Radar de Riesgo y Anomalías (Vista de Auditoría):")
df_analisis.select(columnas_vista).orderBy(F.desc("Anomalia_Deportiva"), F.desc("date")).show(15, truncate=False)

# Cálculos para el Resumen
total_juegos = df_analisis.count()
anomalias = df_analisis.filter(F.col("Anomalia_Deportiva") == "ROMPE-QUINIELAS").count()
volatiles = df_analisis.filter(F.col("Nivel_Riesgo") == "Alta Volatilidad (Evitar)").count()

print("="*60)
print("  REPORTE EJECUTIVO DE DETECCIÓN DE ANOMALÍAS")
print("="*60)
print(f"Juegos validados en este lote: {total_juegos}")
print(f"Descartados por Alta Volatilidad (Riesgo): {volatiles} ({(volatiles/total_juegos)*100:.1f}%)")
print(f"Anomalías Matemáticas ('Rompe-Quinielas'): {anomalias} ({(anomalias/total_juegos)*100:.1f}%)")
print("="*60)

Iniciando Fase 3: Entrenamiento de Inteligencia Artificial (Random Forest)...

Dataset de Entrenamiento: 82 juegos históricos.
Dataset de Validación: 17 juegos.

Ajustando hiperparámetros y entrenando el modelo...
Extrayendo probabilidades y calculando niveles de riesgo...

Radar de Riesgo y Anomalías (Vista de Auditoría):
+------------------------+---------------------+----------+----------+--------------------+------------+----------------+-------------------------+---------------------+
|date                    |home_team_name       |score_home|score_away|away_team_name      |Prob_Local_%|Prob_Visitante_%|Nivel_Riesgo             |Anomalia_Deportiva   |
+------------------------+---------------------+----------+----------+--------------------+------------+----------------+-------------------------+---------------------+
|2026-04-02T18:10:00.000Z|Kansas City Royals   |1         |5         |Minnesota Twins     |66.58       |33.42           |Favorito Matemático      |ROMPE-QUINIELAS   

In [17]:
import pyspark.sql.functions as F

print("---Equipos mas volatiles---  ")

# Filtramos los Rompe-Quinielas, agrupamos por equipo y contamos
df_volatilidad = df_analisis.filter(F.col("Anomalia_Deportiva") == "ROMPE-QUINIELAS") \
    .groupBy("home_team_name") \
    .count() \
    .withColumnRenamed("home_team_name", "Equipo (Local)") \
    .withColumnRenamed("count", "Fallas_Estadisticas") \
    .orderBy(F.desc("Fallas_Estadisticas"))

# Mostramos el Top 5
df_volatilidad.show(5, truncate=False)

---Equipos mas volatiles---  
+---------------------+-------------------+
|Equipo (Local)       |Fallas_Estadisticas|
+---------------------+-------------------+
|Kansas City Royals   |1                  |
|Chicago Cubs         |1                  |
|Toronto Blue Jays    |1                  |
|Philadelphia Phillies|1                  |
+---------------------+-------------------+



In [19]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.ml.functions import vector_to_array

print("Iniciando Fase 5:  Predicción Manual.\n")

# PASO 1: EXTRAER EL MOMENTO ACTUAL

window_latest = Window.partitionBy("home_team_name").orderBy(F.col("date").desc())

df_estado_actual = df_features.withColumn("row_num", F.row_number().over(window_latest)) \
    .filter(F.col("row_num") == 1) \
    .select(
        F.col("home_team_name").alias("Equipo"),
        F.col("Ofensiva_Local").alias("Poder_Ofensivo"),
        F.col("Defensa_Local").alias("Poder_Defensivo"),
        F.col("Promedio_Hits_Local").alias("Hits_Promedio"),
        F.col("Promedio_Errores_Local").alias("Errores_Promedio")
    )


# PASO 2: Aqui le damos de forma manual los partidos a disputar

juegos_de_hoy = [
    ("Toronto Blue Jays", "Boston Red Sox"),
    ("Chicago White Sox", "Los Angeles Angels"),
    ("Pittsburgh Pirates", "St. Louis Cardinals"),
    ("Los Angeles Dodgers", "Miami Marlins"),
    ("San Diego Padres", "Chicago Cubs"),
    ("Texas Rangers", "New York Yankees"),
    ("Minnesota Twins", "Seattle Mariners")
]

print(" Creando cartelera del día...")
df_manual = spark.createDataFrame(juegos_de_hoy, ["home_team_name", "away_team_name"])


# PASO 3: PREPARAR LOS DATOS PARA LA IA

try:
    model_home = indexer_home.fit(df_clean)
    model_away = indexer_away.fit(df_clean)

    df_manual = model_home.transform(df_manual)
    df_manual = model_away.transform(df_manual)

    # 3.2 Inyectar estadísticas del Local
    df_pred = df_manual.join(df_estado_actual, df_manual.home_team_name == df_estado_actual.Equipo, "left") \
        .withColumnRenamed("Poder_Ofensivo", "Ofensiva_Local") \
        .withColumnRenamed("Poder_Defensivo", "Defensa_Local") \
        .withColumnRenamed("Hits_Promedio", "Promedio_Hits_Local") \
        .withColumnRenamed("Errores_Promedio", "Promedio_Errores_Local") \
        .drop("Equipo")

    # 3.3 Inyectar estadísticas del Visitante
    df_pred = df_pred.join(df_estado_actual, df_pred.away_team_name == df_estado_actual.Equipo, "left") \
        .withColumnRenamed("Poder_Ofensivo", "Ofensiva_Visitante") \
        .withColumnRenamed("Poder_Defensivo", "Defensa_Visitante") \
        .withColumnRenamed("Hits_Promedio", "Promedio_Hits_Visitante") \
        .withColumnRenamed("Errores_Promedio", "Promedio_Errores_Visitante") \
        .drop("Equipo")

    df_pred = df_pred.fillna(0)
    df_pred = assembler.transform(df_pred)

    # PASO 4: LA PREDICCIÓN Y EL DETECTOR DE UNDERDOGS

    print("Consultando a la Inteligencia Artificial...\n")
    predicciones_hoy = modelo_entrenado.transform(df_pred)

    df_resultado = predicciones_hoy.withColumn("prob_array", vector_to_array("probability"))
    df_resultado = df_resultado.withColumn("Prob_Visitante_%", F.round(F.col("prob_array")[0] * 100, 2))
    df_resultado = df_resultado.withColumn("Prob_Local_%", F.round(F.col("prob_array")[1] * 100, 2))

    df_resultado = df_resultado.withColumn(
        "Recomendacion_IA",
        F.when((F.col("Prob_Local_%") >= 40) & (F.col("Prob_Local_%") <= 60), "Alta Volatilidad (Evitar)")
         .when(F.col("Prob_Local_%") > 60, "Apunta a Local (Favorito)")
         .otherwise("¡SORPRESA! Apunta a Visitante (Underdog)")
    )

    columnas_finales = ["home_team_name", "away_team_name", "Prob_Visitante_%", "Prob_Local_%", "Recomendacion_IA"]
    print("PRONÓSTICOS DE SÚPER PRECISIÓN (CARTELERA COMPLETA):")

    df_resultado.select(columnas_finales).orderBy(F.asc("Recomendacion_IA")).show(20, truncate=False)

except Exception as e:
    print(f"Ocurrió un error en el procesamiento. Detalle: {e}")

Iniciando Fase 5:  Predicción Manual.

 Creando cartelera del día...
Consultando a la Inteligencia Artificial...

PRONÓSTICOS DE SÚPER PRECISIÓN (CARTELERA COMPLETA):
+-------------------+-------------------+----------------+------------+-------------------------+
|home_team_name     |away_team_name     |Prob_Visitante_%|Prob_Local_%|Recomendacion_IA         |
+-------------------+-------------------+----------------+------------+-------------------------+
|Pittsburgh Pirates |St. Louis Cardinals|6.91            |93.09       |Apunta a Local (Favorito)|
|Toronto Blue Jays  |Boston Red Sox     |36.93           |63.07       |Apunta a Local (Favorito)|
|Chicago White Sox  |Los Angeles Angels |32.38           |67.62       |Apunta a Local (Favorito)|
|San Diego Padres   |Chicago Cubs       |10.53           |89.47       |Apunta a Local (Favorito)|
|Minnesota Twins    |Seattle Mariners   |6.77            |93.23       |Apunta a Local (Favorito)|
|Los Angeles Dodgers|Miami Marlins      |19.16   

In [20]:
# Convertimos los DataFrames de PySpark a Pandas y luego los descargamos como CSV
print("Exportando datos para PowerBI / Tableau...")

df_analisis.toPandas().to_csv('reporte_anomalias.csv', index=False)
df_resultado.toPandas().to_csv('predicciones_de_hoy.csv', index=False)

print("Archivos 'reporte_anomalias.csv' y 'predicciones_de_hoy.csv' guardados.")
# Ve a la carpeta de archivos a la izquierda en Colab y descárgalos a tu PC.

Exportando datos para PowerBI / Tableau...
Archivos 'reporte_anomalias.csv' y 'predicciones_de_hoy.csv' guardados.


In [ ]:
import pandas as pd

print("1. Exportando el Historial de Rendimiento (Features)...")
# Exportamos las variables predictivas clave
columnas_features = ["date", "home_team_name", "Ofensiva_Local", "Defensa_Local", "Promedio_Hits_Local", "Promedio_Errores_Local"]
df_features.select(columnas_features).toPandas().to_csv('historial_rendimiento.csv', index=False)


print(" 2. Extrayendo la 'Caja Blanca' de la IA (Feature Importances)...")
# El VectorAssembler usó estas 10 columnas exactas
nombres_variables = [
    "ID_Local", "ID_Visitante",
    "Ofensiva_Local", "Ofensiva_Visitante",
    "Defensa_Local", "Defensa_Visitante",
    "Promedio_Hits_Local", "Promedio_Hits_Visitante",
    "Promedio_Errores_Local", "Promedio_Errores_Visitante"
]

# Extraemos el peso matemático que le dio el Random Forest a cada una
pesos = modelo_entrenado.featureImportances.toArray()

# Creamos una tabla limpia con Pandas
df_importancia = pd.DataFrame({
    "Variable": nombres_variables,
    "Nivel_de_Importancia_%": pesos * 100
})
# Ordenamos de mayor a menor importancia
df_importancia = df_importancia.sort_values(by="Nivel_de_Importancia_%", ascending=False)

print("\n Ranking de variables que más le importan a tu modelo:")
print(df_importancia.to_string(index=False))

# Lo guardamos para Power BI
df_importancia.to_csv('importancia_variables.csv', index=False)
print("\n Archivos guardados listos para descargar.")

1. Exportando el Historial de Rendimiento (Features)...
 2. Extrayendo la 'Caja Blanca' de la IA (Feature Importances)...

 Ranking de variables que más le importan a tu modelo:
                  Variable  Nivel_de_Importancia_%
              ID_Visitante               26.224487
                  ID_Local               22.791456
    Promedio_Errores_Local               10.140012
         Defensa_Visitante                8.263897
             Defensa_Local                8.155729
        Ofensiva_Visitante                6.475005
       Promedio_Hits_Local                4.860731
   Promedio_Hits_Visitante                4.671459
            Ofensiva_Local                4.515173
Promedio_Errores_Visitante                3.902051

 Archivos guardados listos para descargar.
